# Vibrational Modes — Companion Notebook

**6.7970/8.750 Symmetry and its Application to Machine Learning**

This notebook follows the Vibrational Modes exercise section by section. Use it to **prototype your code** and **test your implementations** against the course library before submitting on the website.

Each section includes small tests you can use to check your work.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atomicarchitects/symm4ml-colabs/blob/main/vib_modes_companion.ipynb)

## Setup

In [1]:
%%capture
!pip install pymatgen
!pip install https://symm4ml.mit.edu/_static/symm4ml_s26/symm4ml/symm4ml_latest.zip

In [2]:
import numpy as np
import itertools
from typing import List
import pprint

from symm4ml import groups, groups_fast, linalg, rep, vib_modes

### Reference data

These vertices, group tables, and irreps are used throughout the exercise for testing.
They are loaded from the course library (`symm4ml.vib_modes`), which uses `pymatgen` to generate symmetry operations.

In [3]:
tetrahedron = np.array([[-1., -1.,  1.],
       [-1.,  1., -1.],
       [ 1., -1., -1.],
       [ 1.,  1.,  1.]])
octahedron = np.array([[-1.,  0.,  0.],
       [ 0., -1.,  0.],
       [ 0.,  0., -1.],
       [ 0.,  0.,  1.],
       [ 0.,  1.,  0.],
       [ 1.,  0.,  0.]])
Td_vec = np.array([[[-1.,  0.,  0.],
        [ 0.,  1.,  0.],
        [ 0.,  0., -1.]],

       [[ 1.,  0.,  0.],
        [ 0.,  1.,  0.],
        [ 0.,  0.,  1.]],

       [[-1.,  0.,  0.],
        [ 0.,  0., -1.],
        [ 0.,  1.,  0.]],

       [[ 0.,  1.,  0.],
        [ 0.,  0.,  1.],
        [ 1.,  0.,  0.]],

       [[ 0.,  0., -1.],
        [ 0.,  1.,  0.],
        [-1.,  0.,  0.]],

       [[ 0.,  1.,  0.],
        [-1.,  0.,  0.],
        [ 0.,  0., -1.]],

       [[ 0.,  1.,  0.],
        [ 1.,  0.,  0.],
        [ 0.,  0.,  1.]],

       [[ 0.,  0.,  1.],
        [ 1.,  0.,  0.],
        [ 0.,  1.,  0.]],

       [[ 0.,  0.,  1.],
        [ 0.,  1.,  0.],
        [ 1.,  0.,  0.]],

       [[ 1.,  0.,  0.],
        [ 0.,  0., -1.],
        [ 0., -1.,  0.]],

       [[-1.,  0.,  0.],
        [ 0., -1.,  0.],
        [ 0.,  0.,  1.]],

       [[ 1.,  0.,  0.],
        [ 0., -1.,  0.],
        [ 0.,  0., -1.]],

       [[ 1.,  0.,  0.],
        [ 0.,  0.,  1.],
        [ 0.,  1.,  0.]],

       [[ 0.,  0., -1.],
        [ 0., -1.,  0.],
        [ 1.,  0.,  0.]],

       [[ 0., -1.,  0.],
        [ 0.,  0.,  1.],
        [-1.,  0.,  0.]],

       [[ 0., -1.,  0.],
        [ 0.,  0., -1.],
        [ 1.,  0.,  0.]],

       [[ 0.,  1.,  0.],
        [ 0.,  0., -1.],
        [-1.,  0.,  0.]],

       [[ 0., -1.,  0.],
        [ 1.,  0.,  0.],
        [ 0.,  0., -1.]],

       [[ 0.,  0.,  1.],
        [-1.,  0.,  0.],
        [ 0., -1.,  0.]],

       [[ 0.,  0., -1.],
        [ 1.,  0.,  0.],
        [ 0., -1.,  0.]],

       [[-1.,  0.,  0.],
        [ 0.,  0.,  1.],
        [ 0., -1.,  0.]],

       [[ 0.,  0.,  1.],
        [ 0., -1.,  0.],
        [-1.,  0.,  0.]],

       [[ 0.,  0., -1.],
        [-1.,  0.,  0.],
        [ 0.,  1.,  0.]],

       [[ 0., -1.,  0.],
        [-1.,  0.,  0.],
        [ 0.,  0.,  1.]]])

Oh_vec = np.array([[[ 0.,  0., -1.],
        [ 0., -1.,  0.],
        [-1.,  0.,  0.]],

       [[ 0.,  0., -1.],
        [-1.,  0.,  0.],
        [ 0.,  1.,  0.]],

       [[ 0.,  0.,  1.],
        [ 1.,  0.,  0.],
        [ 0., -1.,  0.]],

       [[ 0.,  0., -1.],
        [ 0.,  1.,  0.],
        [ 1.,  0.,  0.]],

       [[ 1.,  0.,  0.],
        [ 0.,  0., -1.],
        [ 0., -1.,  0.]],

       [[-1.,  0.,  0.],
        [ 0.,  0.,  1.],
        [ 0., -1.,  0.]],

       [[ 1.,  0.,  0.],
        [ 0.,  1.,  0.],
        [ 0.,  0., -1.]],

       [[ 0.,  1.,  0.],
        [-1.,  0.,  0.],
        [ 0.,  0., -1.]],

       [[-1.,  0.,  0.],
        [ 0.,  1.,  0.],
        [ 0.,  0.,  1.]],

       [[ 1.,  0.,  0.],
        [ 0.,  1.,  0.],
        [ 0.,  0.,  1.]],

       [[ 0.,  1.,  0.],
        [ 1.,  0.,  0.],
        [ 0.,  0.,  1.]],

       [[ 0.,  1.,  0.],
        [ 0.,  0.,  1.],
        [ 1.,  0.,  0.]],

       [[ 0., -1.,  0.],
        [ 1.,  0.,  0.],
        [ 0.,  0.,  1.]],

       [[-1.,  0.,  0.],
        [ 0., -1.,  0.],
        [ 0.,  0., -1.]],

       [[ 0.,  0.,  1.],
        [ 1.,  0.,  0.],
        [ 0.,  1.,  0.]],

       [[ 0.,  0.,  1.],
        [ 0., -1.,  0.],
        [ 1.,  0.,  0.]],

       [[ 0.,  0., -1.],
        [-1.,  0.,  0.],
        [ 0., -1.,  0.]],

       [[ 0.,  1.,  0.],
        [ 0.,  0., -1.],
        [-1.,  0.,  0.]],

       [[ 0.,  0., -1.],
        [ 1.,  0.,  0.],
        [ 0.,  1.,  0.]],

       [[ 0.,  0.,  1.],
        [ 0., -1.,  0.],
        [-1.,  0.,  0.]],

       [[-1.,  0.,  0.],
        [ 0.,  1.,  0.],
        [ 0.,  0., -1.]],

       [[ 0., -1.,  0.],
        [-1.,  0.,  0.],
        [ 0.,  0.,  1.]],

       [[ 0., -1.,  0.],
        [ 0.,  0., -1.],
        [ 1.,  0.,  0.]],

       [[ 0.,  0., -1.],
        [ 0., -1.,  0.],
        [ 1.,  0.,  0.]],

       [[ 0.,  1.,  0.],
        [ 1.,  0.,  0.],
        [ 0.,  0., -1.]],

       [[ 1.,  0.,  0.],
        [ 0., -1.,  0.],
        [ 0.,  0., -1.]],

       [[ 0.,  0.,  1.],
        [ 0.,  1.,  0.],
        [ 1.,  0.,  0.]],

       [[-1.,  0.,  0.],
        [ 0.,  0.,  1.],
        [ 0.,  1.,  0.]],

       [[-1.,  0.,  0.],
        [ 0., -1.,  0.],
        [ 0.,  0.,  1.]],

       [[ 0., -1.,  0.],
        [-1.,  0.,  0.],
        [ 0.,  0., -1.]],

       [[ 1.,  0.,  0.],
        [ 0.,  0.,  1.],
        [ 0.,  1.,  0.]],

       [[ 1.,  0.,  0.],
        [ 0., -1.,  0.],
        [ 0.,  0.,  1.]],

       [[ 1.,  0.,  0.],
        [ 0.,  0.,  1.],
        [ 0., -1.,  0.]],

       [[ 0., -1.,  0.],
        [ 0.,  0., -1.],
        [-1.,  0.,  0.]],

       [[ 0.,  0.,  1.],
        [ 0.,  1.,  0.],
        [-1.,  0.,  0.]],

       [[ 0., -1.,  0.],
        [ 0.,  0.,  1.],
        [-1.,  0.,  0.]],

       [[ 0.,  0., -1.],
        [ 1.,  0.,  0.],
        [ 0., -1.,  0.]],

       [[ 0.,  1.,  0.],
        [ 0.,  0., -1.],
        [ 1.,  0.,  0.]],

       [[-1.,  0.,  0.],
        [ 0.,  0., -1.],
        [ 0.,  1.,  0.]],

       [[ 0.,  1.,  0.],
        [ 0.,  0.,  1.],
        [-1.,  0.,  0.]],

       [[-1.,  0.,  0.],
        [ 0.,  0., -1.],
        [ 0., -1.,  0.]],

       [[ 0., -1.,  0.],
        [ 0.,  0.,  1.],
        [ 1.,  0.,  0.]],

       [[ 0.,  1.,  0.],
        [-1.,  0.,  0.],
        [ 0.,  0.,  1.]],

       [[ 0.,  0.,  1.],
        [-1.,  0.,  0.],
        [ 0., -1.,  0.]],

       [[ 0.,  0., -1.],
        [ 0.,  1.,  0.],
        [-1.,  0.,  0.]],

       [[ 0.,  0.,  1.],
        [-1.,  0.,  0.],
        [ 0.,  1.,  0.]],

       [[ 1.,  0.,  0.],
        [ 0.,  0., -1.],
        [ 0.,  1.,  0.]],

       [[ 0., -1.,  0.],
        [ 1.,  0.,  0.],
        [ 0.,  0., -1.]]])

C2v_vec = np.array([[[-1.,  0.,  0.],
        [ 0.,  1.,  0.],
        [ 0.,  0.,  1.]],

       [[ 1.,  0.,  0.],
        [ 0., -1.,  0.],
        [ 0.,  0.,  1.]],

       [[ 1.,  0.,  0.],
        [ 0.,  1.,  0.],
        [ 0.,  0.,  1.]],

       [[-1.,  0.,  0.],
        [ 0., -1.,  0.],
        [ 0.,  0.,  1.]]])

Td_table = np.array([
    [ 1,  0,  9, 14,  8, 23, 17, 19,  4,  2, 11, 10, 20, 21,  3, 16,
        15,  6, 22,  7, 12, 13, 18,  5],
       [ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15,
        16, 17, 18, 19, 20, 21, 22, 23],
       [12,  2, 11, 23,  7, 14, 15, 13, 22,  0,  9, 20, 10, 18,  6,  5,
        17,  3,  4,  8,  1, 19, 21, 16],
       [16,  3,  4,  7,  5,  2, 12,  1,  6, 13, 14, 15,  8, 17, 18, 19,
        22,  9, 10, 11, 21, 23,  0, 20],
       [ 8,  4, 15, 20,  1, 18, 19, 17,  0, 16, 13, 21, 14, 10, 12,  2,
         9,  7,  5,  6,  3, 11, 23, 22],
       [ 6,  5, 19, 21,  3, 10, 11,  9, 16, 22, 17, 23, 18, 14,  8,  4,
        13,  1,  2, 12,  7, 15, 20,  0],
       [ 5,  6, 22,  8, 16,  0,  1, 12,  3, 19, 23, 17,  7, 15, 21, 13,
         4, 11, 20,  9, 18, 14,  2, 10],
       [22,  7,  5,  1,  2,  4,  8,  3, 12, 17, 18, 19,  6,  9, 10, 11,
         0, 13, 14, 15, 23, 20, 16, 21],
       [ 4,  8, 16, 12,  0, 22,  7,  6,  1, 15, 21, 13,  3, 11, 20,  9,
         2, 19, 23, 17, 14, 10,  5, 18],
       [20,  9, 10,  5, 19,  3, 16, 21, 18,  1,  2, 12, 11, 22, 17, 23,
         6, 14,  8,  4,  0,  7, 13, 15],
       [11, 10, 12, 15, 21, 17, 23, 22, 13, 20,  1,  0,  2,  8, 16,  3,
        14,  5, 19, 18,  9,  4,  7,  6],
       [10, 11, 20, 16, 13,  6,  5, 18, 21, 12,  0,  1,  9,  4, 15, 14,
         3, 23,  7, 22,  2,  8, 19, 17],
       [ 2, 12,  0,  6, 22, 16,  3,  8,  7, 11, 20,  9,  1, 19, 23, 17,
         5, 15, 21, 13, 10, 18,  4, 14],
       [21, 13, 14,  2, 11,  7, 22, 23, 10,  3,  4,  8, 15,  0,  9, 20,
        12, 18,  6,  5, 16,  1, 17, 19],
       [15, 14,  8, 19, 23,  9, 20,  0, 17, 21,  3, 16,  4,  6, 22,  7,
        18,  2, 11, 10, 13,  5,  1, 12],
       [14, 15, 21, 22, 17, 12,  2, 10, 23,  8, 16,  3, 13,  5, 19, 18,
         7, 20,  1,  0,  4,  6, 11,  9],
       [ 3, 16, 13, 18,  6, 20,  9, 11,  5,  4, 15, 14, 21, 23,  7, 22,
        19, 12,  0,  1,  8, 17, 10,  2],
       [23, 17, 18,  4, 15,  1,  0, 20, 14,  7,  5,  6, 19, 16, 13, 21,
         8, 10, 12,  2, 22,  3,  9, 11],
       [19, 18,  6, 11, 20, 13, 21, 16,  9, 23,  7, 22,  5, 12,  0,  1,
        10,  4, 15, 14, 17,  2,  3,  8],
       [18, 19, 23,  0,  9,  8,  4, 14, 20,  6, 22,  7, 17,  2, 11, 10,
         1, 21,  3, 16,  5, 12, 15, 13],
       [ 9, 20,  1, 17, 18, 15, 14,  4, 19, 10, 12,  2,  0,  7,  5,  6,
        23, 16, 13, 21, 11, 22,  8,  3],
       [13, 21,  3,  9, 10, 19, 18,  5, 11, 14,  8,  4, 16,  1,  2, 12,
        20, 22, 17, 23, 15,  0,  6,  7],
       [ 7, 22, 17, 10, 12, 21, 13, 15,  2,  5, 19, 18, 23, 20,  1,  0,
        11,  8, 16,  3,  6,  9, 14,  4],
       [17, 23,  7, 13, 14, 11, 10,  2, 15, 18,  6,  5, 22,  3,  4,  8,
        21,  0,  9, 20, 19, 16, 12,  1]])

Oh_table = np.array([
    [ 9, 12,  7, ..., 47, 35, 45],
       [32, 35, 37, ..., 33, 29, 26],
       [38, 37, 35, ..., 11, 10,  0],
       ...,
       [ 5, 39, 22, ..., 17,  7,  3],
       [36,  0, 26, ..., 19, 25, 41],
       [37,  4, 27, ..., 32,  2, 28]])
C2v_table = np.array(
    [[2, 3, 0, 1],
       [3, 2, 1, 0],
       [0, 1, 2, 3],
       [1, 0, 3, 2]])

Td_irreps = [np.array([[[1.+0.j]],
        [[1.+0.j]],
        [[1.+0.j]],
        [[1.+0.j]],
        [[1.+0.j]],
        [[1.+0.j]],
        [[1.+0.j]],
        [[1.+0.j]],
        [[1.+0.j]],
        [[1.+0.j]],
        [[1.+0.j]],
        [[1.+0.j]],
        [[1.+0.j]],
        [[1.+0.j]],
        [[1.+0.j]],
        [[1.+0.j]],
        [[1.+0.j]],
        [[1.+0.j]],
        [[1.+0.j]],
        [[1.+0.j]],
        [[1.+0.j]],
        [[1.+0.j]],
        [[1.+0.j]],
        [[1.+0.j]]]),
 np.array([[[ 1.00000000e+00+0.j,  4.57225380e-16+0.j],
         [ 4.50302277e-16+0.j,  1.00000000e+00+0.j]],
        [[ 1.00000000e+00+0.j,  4.71580045e-16+0.j],
         [ 4.71580045e-16+0.j,  1.00000000e+00+0.j]],
        [[ 9.54839059e-01+0.j,  2.97123496e-01+0.j],
         [ 2.97123496e-01+0.j, -9.54839059e-01+0.j]],
        [[-5.00000000e-01+0.j, -8.66025404e-01+0.j],
         [ 8.66025404e-01+0.j, -5.00000000e-01+0.j]],
        [[-7.34736025e-01+0.j,  6.78353134e-01+0.j],
         [ 6.78353134e-01+0.j,  7.34736025e-01+0.j]],
        [[-2.20103034e-01+0.j, -9.75476629e-01+0.j],
         [-9.75476629e-01+0.j,  2.20103034e-01+0.j]],
        [[-2.20103034e-01+0.j, -9.75476629e-01+0.j],
         [-9.75476629e-01+0.j,  2.20103034e-01+0.j]],
        [[-5.00000000e-01+0.j,  8.66025404e-01+0.j],
         [-8.66025404e-01+0.j, -5.00000000e-01+0.j]],
        [[-7.34736025e-01+0.j,  6.78353134e-01+0.j],
         [ 6.78353134e-01+0.j,  7.34736025e-01+0.j]],
        [[ 9.54839059e-01+0.j,  2.97123496e-01+0.j],
         [ 2.97123496e-01+0.j, -9.54839059e-01+0.j]],
        [[ 1.00000000e+00+0.j,  4.60656430e-16+0.j],
         [ 4.71176595e-16+0.j,  1.00000000e+00+0.j]],
        [[ 1.00000000e+00+0.j,  4.59456179e-16+0.j],
         [ 4.81354393e-16+0.j,  1.00000000e+00+0.j]],
        [[ 9.54839059e-01+0.j,  2.97123496e-01+0.j],
         [ 2.97123496e-01+0.j, -9.54839059e-01+0.j]],
        [[-7.34736025e-01+0.j,  6.78353134e-01+0.j],
         [ 6.78353134e-01+0.j,  7.34736025e-01+0.j]],
        [[-5.00000000e-01+0.j, -8.66025404e-01+0.j],
         [ 8.66025404e-01+0.j, -5.00000000e-01+0.j]],
        [[-5.00000000e-01+0.j, -8.66025404e-01+0.j],
         [ 8.66025404e-01+0.j, -5.00000000e-01+0.j]],
        [[-5.00000000e-01+0.j, -8.66025404e-01+0.j],
         [ 8.66025404e-01+0.j, -5.00000000e-01+0.j]],
        [[-2.20103034e-01+0.j, -9.75476629e-01+0.j],
         [-9.75476629e-01+0.j,  2.20103034e-01+0.j]],
        [[-5.00000000e-01+0.j,  8.66025404e-01+0.j],
         [-8.66025404e-01+0.j, -5.00000000e-01+0.j]],
        [[-5.00000000e-01+0.j,  8.66025404e-01+0.j],
         [-8.66025404e-01+0.j, -5.00000000e-01+0.j]],
        [[ 9.54839059e-01+0.j,  2.97123496e-01+0.j],
         [ 2.97123496e-01+0.j, -9.54839059e-01+0.j]],
        [[-7.34736025e-01+0.j,  6.78353134e-01+0.j],
         [ 6.78353134e-01+0.j,  7.34736025e-01+0.j]],
        [[-5.00000000e-01+0.j,  8.66025404e-01+0.j],
         [-8.66025404e-01+0.j, -5.00000000e-01+0.j]],
        [[-2.20103034e-01+0.j, -9.75476629e-01+0.j],
         [-9.75476629e-01+0.j,  2.20103034e-01+0.j]]]),
 np.array([[[ 1.+0.j]],
        [[ 1.+0.j]],
        [[-1.+0.j]],
        [[ 1.+0.j]],
        [[-1.+0.j]],
        [[-1.+0.j]],
        [[-1.+0.j]],
        [[ 1.+0.j]],
        [[-1.+0.j]],
        [[-1.+0.j]],
        [[ 1.+0.j]],
        [[ 1.+0.j]],
        [[-1.+0.j]],
        [[-1.+0.j]],
        [[ 1.+0.j]],
        [[ 1.+0.j]],
        [[ 1.+0.j]],
        [[-1.+0.j]],
        [[ 1.+0.j]],
        [[ 1.+0.j]],
        [[-1.+0.j]],
        [[-1.+0.j]],
        [[ 1.+0.j]],
        [[-1.+0.j]]]),
 np.array([[[-9.23518155e-01+0.j,  3.83554713e-01+0.j,  1.03314950e-15+0.j],
         [ 3.83554713e-01+0.j,  9.23518155e-01+0.j, -7.15376644e-15+0.j],
         [ 1.04561787e-15+0.j, -7.16423831e-15+0.j, -1.00000000e+00+0.j]],
        [[ 1.00000000e+00+0.j,  4.14561599e-16+0.j, -4.43479708e-15+0.j],
         [ 4.14561599e-16+0.j,  1.00000000e+00+0.j, -9.87737154e-15+0.j],
         [-4.43479708e-15+0.j, -9.87737154e-15+0.j,  1.00000000e+00+0.j]],
        [[-9.03560237e-01+0.j,  4.26166070e-01+0.j, -4.42874603e-02+0.j],
         [-6.58213694e-02+0.j, -3.59268531e-02+0.j,  9.97184441e-01+0.j],
         [-4.23375066e-01+0.j, -9.03931270e-01+0.j, -6.05129103e-02+0.j]],
        [[ 4.62377057e-01+0.j,  8.77072189e-01+0.j, -1.30199205e-01+0.j],
         [ 1.53794487e-01+0.j, -2.23942374e-01+0.j, -9.62391328e-01+0.j],
         [-8.73243788e-01+0.j,  4.24963750e-01+0.j, -2.38434683e-01+0.j]],
        [[-4.20392518e-01+0.j,  2.83230103e-01+0.j,  8.62003967e-01+0.j],
         [ 2.83230103e-01+0.j,  9.43523153e-01+0.j, -1.71885918e-01+0.j],
         [ 8.62003967e-01+0.j, -1.71885918e-01+0.j,  4.76869366e-01+0.j]],
        [[-5.81988407e-02+0.j,  9.80876428e-01+0.j,  1.85726486e-01+0.j],
         [-9.57666416e-01+0.j, -2.31406952e-03+0.j, -2.87870944e-01+0.j],
         [ 2.81936039e-01+0.j,  1.94617774e-01+0.j, -9.39487090e-01+0.j]],
        [[ 4.29967463e-01+0.j,  8.83534749e-01+0.j, -1.85726486e-01+0.j],
         [ 8.83534749e-01+0.j, -3.69454553e-01+0.j,  2.87870944e-01+0.j],
         [-1.85726486e-01+0.j,  2.87870944e-01+0.j,  9.39487090e-01+0.j]],
        [[ 4.62377057e-01+0.j,  1.53794487e-01+0.j, -8.73243788e-01+0.j],
         [ 8.77072189e-01+0.j, -2.23942374e-01+0.j,  4.24963750e-01+0.j],
         [-1.30199205e-01+0.j, -9.62391328e-01+0.j, -2.38434683e-01+0.j]],
        [[ 4.96874364e-01+0.j,  1.00324610e-01+0.j, -8.62003967e-01+0.j],
         [ 1.00324610e-01+0.j,  9.79995002e-01+0.j,  1.71885918e-01+0.j],
         [-8.62003967e-01+0.j,  1.71885918e-01+0.j, -4.76869366e-01+0.j]],
        [[ 8.09208186e-01+0.j, -4.07352017e-01+0.j,  4.23375066e-01+0.j],
         [-4.07352017e-01+0.j,  1.30278904e-01+0.j,  9.03931270e-01+0.j],
         [ 4.23375066e-01+0.j,  9.03931270e-01+0.j,  6.05129103e-02+0.j]],
        [[-8.83602319e-01+0.j, -2.32100119e-02+0.j, -4.67662526e-01+0.j],
         [-2.32100119e-02+0.j, -9.95371861e-01+0.j,  9.32531702e-02+0.j],
         [-4.67662526e-01+0.j,  9.32531702e-02+0.j,  8.78974179e-01+0.j]],
        [[ 8.07120473e-01+0.j, -3.60344701e-01+0.j,  4.67662526e-01+0.j],
         [-3.60344701e-01+0.j, -9.28146294e-01+0.j, -9.32531702e-02+0.j],
         [ 4.67662526e-01+0.j, -9.32531702e-02+0.j, -8.78974179e-01+0.j]],
        [[ 9.97912287e-01+0.j,  4.70073158e-02+0.j,  4.42874603e-02+0.j],
         [ 4.70073158e-02+0.j, -5.84251975e-02+0.j, -9.97184441e-01+0.j],
         [ 4.42874603e-02+0.j, -9.97184441e-01+0.j,  6.05129103e-02+0.j]],
        [[-3.82409226e-02+0.j, -1.91777357e-01+0.j,  9.80693162e-01+0.j],
         [-1.91777357e-01+0.j, -9.61759077e-01+0.j, -1.95552864e-01+0.j],
         [-9.80693162e-01+0.j,  1.95552864e-01+0.j,  4.05289967e-18+0.j]],
        [[-3.68025006e-01+0.j, -8.95886242e-01+0.j, -2.48888400e-01+0.j],
         [ 3.19378900e-01+0.j,  1.29590323e-01+0.j, -9.38724383e-01+0.j],
         [ 8.73243788e-01+0.j, -4.24963750e-01+0.j,  2.38434683e-01+0.j]],
        [[-3.74361588e-03+0.j, -9.68524935e-01+0.j,  2.48888400e-01+0.j],
         [-2.45247233e-01+0.j,  2.42178299e-01+0.j,  9.38724383e-01+0.j],
         [-9.69453341e-01+0.j, -5.75249679e-02+0.j, -2.38434683e-01+0.j]],
        [[-9.06084348e-02+0.j,  9.87338989e-01+0.j,  1.30199205e-01+0.j],
         [-2.27926153e-01+0.j, -1.47826248e-01+0.j,  9.62391328e-01+0.j],
         [ 9.69453341e-01+0.j,  5.75249679e-02+0.j,  2.38434683e-01+0.j]],
        [[-5.81988407e-02+0.j, -9.57666416e-01+0.j,  2.81936039e-01+0.j],
         [ 9.80876428e-01+0.j, -2.31406952e-03+0.j,  1.94617774e-01+0.j],
         [ 1.85726486e-01+0.j, -2.87870944e-01+0.j, -9.39487090e-01+0.j]],
        [[-3.74361588e-03+0.j, -2.45247233e-01+0.j, -9.69453341e-01+0.j],
         [-9.68524935e-01+0.j,  2.42178299e-01+0.j, -5.75249679e-02+0.j],
         [ 2.48888400e-01+0.j,  9.38724383e-01+0.j, -2.38434683e-01+0.j]],
        [[-9.06084348e-02+0.j, -2.27926153e-01+0.j,  9.69453341e-01+0.j],
         [ 9.87338989e-01+0.j, -1.47826248e-01+0.j,  5.75249679e-02+0.j],
         [ 1.30199205e-01+0.j,  9.62391328e-01+0.j,  2.38434683e-01+0.j]],
        [[-9.03560237e-01+0.j, -6.58213694e-02+0.j, -4.23375066e-01+0.j],
         [ 4.26166070e-01+0.j, -3.59268531e-02+0.j, -9.03931270e-01+0.j],
         [-4.42874603e-02+0.j,  9.97184441e-01+0.j, -6.05129103e-02+0.j]],
        [[-3.82409226e-02+0.j, -1.91777357e-01+0.j, -9.80693162e-01+0.j],
         [-1.91777357e-01+0.j, -9.61759077e-01+0.j,  1.95552864e-01+0.j],
         [ 9.80693162e-01+0.j, -1.95552864e-01+0.j,  1.60902412e-19+0.j]],
        [[-3.68025006e-01+0.j,  3.19378900e-01+0.j,  8.73243788e-01+0.j],
         [-8.95886242e-01+0.j,  1.29590323e-01+0.j, -4.24963750e-01+0.j],
         [-2.48888400e-01+0.j, -9.38724383e-01+0.j,  2.38434683e-01+0.j]],
        [[-3.13569781e-01+0.j, -9.06744761e-01+0.j, -2.81936039e-01+0.j],
         [-9.06744761e-01+0.j,  3.74082692e-01+0.j, -1.94617774e-01+0.j],
         [-2.81936039e-01+0.j, -1.94617774e-01+0.j,  9.39487090e-01+0.j]]]),
 np.array([[[-7.12834780e-01+2.46519033e-32j,
           7.01332002e-01+3.11089333e-16j,
          -1.68909439e-15+8.20285192e-17j],
         [ 7.01332002e-01-3.11089333e-16j,
           7.12834780e-01+0.00000000e+00j,
          -4.25292894e-15+2.00334935e-16j],
         [-1.68637346e-15-8.20285192e-17j,
          -4.25799235e-15-2.00334935e-16j,
          -1.00000000e+00+0.00000000e+00j]],
        [[ 1.00000000e+00+0.00000000e+00j,
           1.78846330e-16-1.47911420e-31j,
          -2.26965632e-17+9.86076132e-32j],
         [ 1.78846330e-16+1.47911420e-31j,
           1.00000000e+00+0.00000000e+00j,
           7.69327704e-17+6.16297582e-32j],
         [-2.26965632e-17-9.86076132e-32j,
           7.69327704e-17-6.16297582e-32j,
           1.00000000e+00+0.00000000e+00j]],
        [[ 7.41297541e-01-3.26347141e-16j,
           6.31043025e-02-3.81862303e-16j,
          -6.68203415e-01-1.30688658e-17j],
         [-6.70163074e-01-3.87482909e-17j,
           1.24282198e-01+1.24943420e-16j,
          -7.31734508e-01-3.01778766e-16j],
         [ 3.68701935e-02-3.45490133e-16j,
           9.90238245e-01-1.95584042e-17j,
           1.34420261e-01+2.01403721e-16j]],
        [[-9.44452516e-02-1.55091864e-16j,
           9.69036638e-01-7.90101433e-17j,
          -2.28140501e-01-2.73233694e-16j],
         [-3.27962429e-01-1.46408817e-16j,
          -2.46658039e-01+2.82617763e-16j,
          -9.11921300e-01-1.39501020e-16j],
         [-9.39957839e-01+6.60492963e-17j,
          -1.13051237e-02+3.15946166e-17j,
           3.41103290e-01-1.27525899e-16j]],
        [[-7.27836189e-01-2.46519033e-32j,
          -1.11439348e-01-1.28571075e-16j,
          -6.76635614e-01-2.12429518e-16j],
         [-1.11439348e-01+1.28571075e-16j,
          -9.54370391e-01+0.00000000e+00j,
           2.77053114e-01-2.32664150e-16j],
         [-6.76635614e-01+2.12429518e-16j,
           2.77053114e-01+2.32664150e-16j,
           6.82206580e-01+0.00000000e+00j]],
        [[ 1.15119849e-01+3.82692265e-16j,
          -9.77501995e-01-2.39491565e-17j,
           1.76740688e-01-3.08748278e-16j],
         [ 8.83228764e-01-5.59738363e-17j,
           1.93004120e-02-4.62060426e-16j,
          -4.68545030e-01+2.09254420e-16j],
         [ 4.54592533e-01+1.05701509e-16j,
           2.10041293e-01+1.27368993e-16j,
           8.65579739e-01+7.93681609e-17j]],
        [[-7.67614863e-01+2.46519033e-32j,
          -6.16060185e-01+2.51727764e-16j,
          -1.76740688e-01+1.22363591e-16j],
         [-6.16060185e-01-2.51727764e-16j,
           6.33194602e-01+0.00000000e+00j,
           4.68545030e-01-1.32937926e-16j],
         [-1.76740688e-01-1.22363591e-16j,
           4.68545030e-01+1.32937926e-16j,
          -8.65579739e-01+9.24446373e-33j]],
        [[-9.44452516e-02+1.55091864e-16j,
          -3.27962429e-01+1.46408817e-16j,
          -9.39957839e-01-6.60492963e-17j],
         [ 9.69036638e-01+7.90101433e-17j,
          -2.46658039e-01-2.82617763e-16j,
          -1.13051237e-02-3.15946166e-17j],
         [-2.28140501e-01+2.73233694e-16j,
          -9.11921300e-01+1.39501020e-16j,
           3.41103290e-01+1.27525899e-16j]],
        [[ 4.40670969e-01+0.00000000e+00j,
          -5.89892654e-01-1.82518257e-16j,
           6.76635614e-01+1.30400998e-16j],
         [-5.89892654e-01+1.82518257e-16j,
          -7.58464388e-01+0.00000000e+00j,
          -2.77053114e-01+3.23292153e-17j],
         [ 6.76635614e-01-1.30400998e-16j,
          -2.77053114e-01-3.23292153e-17j,
          -6.82206580e-01-4.93038066e-32j]],
        [[-9.98429479e-01-6.35556882e-33j,
           4.21801414e-02+4.79722193e-16j,
          -3.68701935e-02-4.18939669e-16j],
         [ 4.21801414e-02-4.79722193e-16j,
           1.32849740e-01-6.16297582e-33j,
          -9.90238245e-01+1.05160144e-17j],
         [-3.68701935e-02+4.18939669e-16j,
          -9.90238245e-01-1.05160144e-17j,
          -1.34420261e-01+0.00000000e+00j]],
        [[-7.69760302e-01+0.00000000e+00j,
          -9.42732309e-02+3.20246798e-17j,
           6.31333221e-01-4.14449787e-16j],
         [-9.42732309e-02-3.20246798e-17j,
          -9.61399176e-01+0.00000000e+00j,
          -2.58503737e-01+8.18854276e-17j],
         [ 6.31333221e-01+4.14449787e-16j,
          -2.58503737e-01-8.18854276e-17j,
           7.31159478e-01-1.23259516e-32j]],
        [[ 4.82595081e-01+5.63334509e-33j,
          -6.07058771e-01-3.43114012e-16j,
          -6.31333221e-01+3.32421267e-16j],
         [-6.07058771e-01+3.43114012e-16j,
          -7.51435604e-01+2.46519033e-32j,
           2.58503737e-01-2.82220362e-16j],
         [-6.31333221e-01-3.32421267e-16j,
           2.58503737e-01+2.82220362e-16j,
          -7.31159478e-01-1.23259516e-32j]],
        [[-4.84165602e-01+6.16297582e-33j,
           5.64878630e-01-1.36608180e-16j,
           6.68203415e-01+8.65184017e-17j],
         [ 5.64878630e-01+1.36608180e-16j,
          -3.81414137e-01+1.23259516e-32j,
           7.31734508e-01+2.71704348e-16j],
         [ 6.68203415e-01-8.65184017e-17j,
           7.31734508e-01-2.71704348e-16j,
          -1.34420261e-01-2.46519033e-32j]],
        [[ 1.43582610e-01-3.98855677e-16j,
           3.50666001e-01+2.10619826e-16j,
          -9.25428220e-01+1.70442543e-16j],
         [ 3.50666001e-01-1.00469507e-16j,
           8.56417390e-01+2.17683194e-17j,
           3.78922961e-01-1.20906421e-16j],
         [ 9.25428220e-01+8.84140237e-17j,
          -3.78922961e-01-3.21241355e-16j,
           1.25314611e-17+3.77087358e-16j]],
        [[-1.62686687e-01-1.71255277e-16j,
          -8.63752194e-01+1.76870033e-16j,
          -4.76933107e-01-1.58774840e-16j],
         [-3.00020503e-01-3.72061666e-16j,
           5.03789977e-01-1.57674343e-16j,
          -8.10051453e-01-1.51761732e-16j],
         [ 9.39957839e-01+7.40023955e-18j,
           1.13051237e-02-6.16690352e-17j,
          -3.41103290e-01+3.28929620e-16j]],
        [[-4.89808327e-01+5.53947542e-16j,
          -7.29809985e-01+5.09085746e-17j,
           4.76933107e-01-2.76098469e-17j],
         [ 5.67189082e-01+6.43600666e-17j,
           1.48705037e-01-3.04386083e-16j,
           8.10051453e-01+2.28078226e-16j],
         [-6.62105994e-01-2.40623216e-17j,
           6.67281199e-01+3.21975954e-16j,
           3.41103290e-01-2.49561459e-16j]],
        [[ 7.46940266e-01-2.27600401e-16j,
           6.24525541e-01-1.48768464e-16j,
           2.28140501e-01+4.59618382e-16j],
         [ 6.07938498e-02+4.54110417e-16j,
          -4.05836975e-01+1.79442663e-16j,
           9.11921300e-01+6.31845260e-17j],
         [ 6.62105994e-01-4.93872142e-17j,
          -6.67281199e-01-2.91901535e-16j,
          -3.41103290e-01+4.81577377e-17j]],
        [[ 1.15119849e-01-3.82692265e-16j,
           8.83228764e-01+5.59738363e-17j,
           4.54592533e-01-1.05701509e-16j],
         [-9.77501995e-01+2.39491565e-17j,
           1.93004120e-02+4.62060426e-16j,
           2.10041293e-01-1.27368993e-16j],
         [ 1.76740688e-01+3.08748278e-16j,
          -4.68545030e-01-2.09254420e-16j,
           8.65579739e-01-7.93681609e-17j]],
        [[-4.89808327e-01-5.53947542e-16j,
           5.67189082e-01-6.43600666e-17j,
          -6.62105994e-01+2.40623216e-17j],
         [-7.29809985e-01-5.09085746e-17j,
           1.48705037e-01+3.04386083e-16j,
           6.67281199e-01-3.21975954e-16j],
         [ 4.76933107e-01+2.76098469e-17j,
           8.10051453e-01-2.28078226e-16j,
           3.41103290e-01+2.49561459e-16j]],
        [[ 7.46940266e-01+2.27600401e-16j,
           6.07938498e-02-4.54110417e-16j,
           6.62105994e-01+4.93872142e-17j],
         [ 6.24525541e-01+1.48768464e-16j,
          -4.05836975e-01-1.79442663e-16j,
          -6.67281199e-01+2.91901535e-16j],
         [ 2.28140501e-01-4.59618382e-16j,
           9.11921300e-01-6.31845260e-17j,
          -3.41103290e-01-4.81577377e-17j]],
        [[ 7.41297541e-01+3.26347141e-16j,
          -6.70163074e-01+3.87482909e-17j,
           3.68701935e-02+3.45490133e-16j],
         [ 6.31043025e-02+3.81862303e-16j,
           1.24282198e-01-1.24943420e-16j,
           9.90238245e-01+1.95584042e-17j],
         [-6.68203415e-01+1.30688658e-17j,
          -7.31734508e-01+3.01778766e-16j,
           1.34420261e-01-2.01403721e-16j]],
        [[ 1.43582610e-01+3.98855677e-16j,
           3.50666001e-01+1.00469507e-16j,
           9.25428220e-01-8.84140237e-17j],
         [ 3.50666001e-01-2.10619826e-16j,
           8.56417390e-01-2.17683194e-17j,
          -3.78922961e-01+3.21241355e-16j],
         [-9.25428220e-01-1.70442543e-16j,
           3.78922961e-01+1.20906421e-16j,
           1.00061637e-17-3.77087358e-16j]],
        [[-1.62686687e-01+1.71255277e-16j,
          -3.00020503e-01+3.72061666e-16j,
           9.39957839e-01-7.40023955e-18j],
         [-8.63752194e-01-1.76870033e-16j,
           5.03789977e-01+1.57674343e-16j,
           1.13051237e-02+6.16690352e-17j],
         [-4.76933107e-01+1.58774840e-16j,
          -8.10051453e-01+1.51761732e-16j,
          -3.41103290e-01-3.28929620e-16j]],
        [[ 5.37375165e-01+0.00000000e+00j,
           7.10333415e-01-2.83752443e-16j,
          -4.54592533e-01+2.92086196e-16j],
         [ 7.10333415e-01+2.83752443e-16j,
          -6.71795426e-01+0.00000000e+00j,
          -2.10041293e-01+5.10524984e-17j],
         [-4.54592533e-01-2.92086196e-16j,
          -2.10041293e-01-5.10524984e-17j,
          -8.65579739e-01+0.00000000e+00j]]])]

C2v_irreps = [np.array([[[-1.]],
        [[ 1.]],
        [[ 1.]],
        [[-1.]]]),
 np.array([[[ 1.]],
        [[-1.]],
        [[ 1.]],
        [[-1.]]]),
 np.array([[[-1.]],
        [[-1.]],
        [[ 1.]],
        [[ 1.]]]),
 np.array([[[1.]],
        [[1.]],
        [[1.]],
        [[1.]]])]

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (7,) + inhomogeneous part.

In [4]:
vib_modes.Oh_table[0]

array([ 9, 12,  7, 28, 39, 37, 19,  2, 23,  0, 16, 40,  1, 26, 29, 20, 10,
       32, 21,  6, 15, 18, 27,  8, 43, 34, 13, 22,  3, 14, 33, 44, 17, 30,
       25, 46, 42,  5, 41,  4, 11, 38, 36, 24, 31, 47, 35, 45],
      dtype=int32)

In [3]:
# Vertex coordinates
tetrahedron = vib_modes.tetrahedron
octahedron = vib_modes.octahedron

# Group vector representations (3x3 rotation/reflection matrices)
Td_vec = vib_modes.Td_vec
Oh_vec = vib_modes.Oh_vec
C2v_vec = vib_modes.C2v_vec

# Multiplication tables
Td_table = vib_modes.Td_table
Oh_table = vib_modes.Oh_table
C2v_table = vib_modes.C2v_table

# Irreps (precomputed with fixed random seed)
Td_irreps = vib_modes.Td_irreps
C2v_irreps = vib_modes.C2v_irreps

# Oh_irreps is not stored in the module; compute it here
np.random.seed(42)
Oh_irreps = rep.infer_irreps(Oh_table)

print(f"Td: {len(Td_vec)} elements, {len(Td_irreps)} irreps")
print(f"Oh: {len(Oh_vec)} elements, {len(Oh_irreps)} irreps")
print(f"C2v: {len(C2v_vec)} elements, {len(C2v_irreps)} irreps")
print(f"Tetrahedron: {tetrahedron.shape[0]} vertices")
print(f"Octahedron: {octahedron.shape[0]} vertices")

Td: 24 elements, 5 irreps
Oh: 48 elements, 10 irreps
C2v: 4 elements, 4 irreps
Tetrahedron: 4 vertices
Octahedron: 6 vertices


---
## Section 1: Degrees of Freedom and Subspaces

A system of $N$ points in 3D has $3N$ degrees of freedom. We remove translational (3) and rotational (up to 3) degrees to isolate the vibrational modes.

### 1.1 `translation_projector(n, d=3)`

Return a projector for the translational degrees of freedom for $n$ vertices in $d$ dimensions.

**Hint:** The translation modes are uniform displacements along each spatial axis. Build an orthonormal basis for these and project.

In [4]:
def translation_projector(n: int, d: int = 3) -> np.ndarray:
    """Projector of translation degrees of freedom for n objects in d dimensions.

    Input:
        n (int): Number of objects
        d (int): Dimensionality of the space (default: 3)
    Output:
        (n * 3, n * 3) np.array projector of translation degrees of freedom
    """
    # YOUR CODE HERE
    d_vec = np.eye(d)
    vec_list = []
    for vals in itertools.product(d_vec, repeat=n):
        vec = []
        for val in vals:
            vec = vec + val.tolist()
        vec_list.append(vec)
    q,p = linalg.gram_schmidt(np.stack(vec_list))
    return q

In [27]:
d_vec = np.eye(3)
vec_list = []
for pair in itertools.product(d_vec, repeat=2):
    vec_list.append(pair[0].tolist() + pair[1].tolist())
    print(pair[0].tolist() + pair[1].tolist())
print(np.stack(vec_list))

[1.0, 0.0, 0.0, 1.0, 0.0, 0.0]
[1.0, 0.0, 0.0, 0.0, 1.0, 0.0]
[1.0, 0.0, 0.0, 0.0, 0.0, 1.0]
[0.0, 1.0, 0.0, 1.0, 0.0, 0.0]
[0.0, 1.0, 0.0, 0.0, 1.0, 0.0]
[0.0, 1.0, 0.0, 0.0, 0.0, 1.0]
[0.0, 0.0, 1.0, 1.0, 0.0, 0.0]
[0.0, 0.0, 1.0, 0.0, 1.0, 0.0]
[0.0, 0.0, 1.0, 0.0, 0.0, 1.0]
[[1. 0. 0. 1. 0. 0.]
 [1. 0. 0. 0. 1. 0.]
 [1. 0. 0. 0. 0. 1.]
 [0. 1. 0. 1. 0. 0.]
 [0. 1. 0. 0. 1. 0.]
 [0. 1. 0. 0. 0. 1.]
 [0. 0. 1. 1. 0. 0.]
 [0. 0. 1. 0. 1. 0.]
 [0. 0. 1. 0. 0. 1.]]


In [31]:
translation_projector(2,3)

array([[ 0.70710678,  0.        ,  0.        ,  0.70710678,  0.        ,
         0.        ],
       [ 0.40824829,  0.        ,  0.        , -0.40824829,  0.81649658,
         0.        ],
       [ 0.28867513,  0.        ,  0.        , -0.28867513, -0.28867513,
         0.8660254 ],
       [-0.2236068 ,  0.89442719,  0.        ,  0.2236068 ,  0.2236068 ,
         0.2236068 ],
       [-0.18257419, -0.18257419,  0.91287093,  0.18257419,  0.18257419,
         0.18257419]])

In [ ]:
array([[ 0.70710678,  0.        ,  0.        ,  0.70710678,  0.        , 0.        ],
       [ 0.40824829,  0.        ,  0.        , -0.40824829,  0.81649658, 0.        ],
       [ 0.28867513,  0.        ,  0.        , -0.28867513, -0.28867513 ,0.8660254 ],
       [-0.2236068 ,  0.89442719,  0.        ,  0.2236068 ,  0.2236068 , 0.2236068 ],
       [-0.18257419, -0.18257419,  0.91287093,  0.18257419,  0.18257419, 0.18257419]])

In [32]:
vib_modes.translation_projector(3,3)

array([[0.33333333, 0.        , 0.        , 0.33333333, 0.        ,
        0.        , 0.33333333, 0.        , 0.        ],
       [0.        , 0.33333333, 0.        , 0.        , 0.33333333,
        0.        , 0.        , 0.33333333, 0.        ],
       [0.        , 0.        , 0.33333333, 0.        , 0.        ,
        0.33333333, 0.        , 0.        , 0.33333333],
       [0.33333333, 0.        , 0.        , 0.33333333, 0.        ,
        0.        , 0.33333333, 0.        , 0.        ],
       [0.        , 0.33333333, 0.        , 0.        , 0.33333333,
        0.        , 0.        , 0.33333333, 0.        ],
       [0.        , 0.        , 0.33333333, 0.        , 0.        ,
        0.33333333, 0.        , 0.        , 0.33333333],
       [0.33333333, 0.        , 0.        , 0.33333333, 0.        ,
        0.        , 0.33333333, 0.        , 0.        ],
       [0.        , 0.33333333, 0.        , 0.        , 0.33333333,
        0.        , 0.        , 0.33333333, 0.        ],


In [ ]:
array([[0.33333333, 0.        , 0.        , 0.33333333, 0.        ,0.        , 0.33333333, 0.        , 0.        ],
       [0.        , 0.33333333, 0.        , 0.        , 0.33333333,0.        , 0.        , 0.33333333, 0.        ],
       [0.        , 0.        , 0.33333333, 0.        , 0.        ,0.33333333, 0.        , 0.        , 0.33333333],
       [0.33333333, 0.        , 0.        , 0.33333333, 0.        ,0.        , 0.33333333, 0.        , 0.        ],
       [0.        , 0.33333333, 0.        , 0.        , 0.33333333,0.        , 0.        , 0.33333333, 0.        ],
       [0.        , 0.        , 0.33333333, 0.        , 0.        ,0.33333333, 0.        , 0.        , 0.33333333],
       [0.33333333, 0.        , 0.        , 0.33333333, 0.        ,0.        , 0.33333333, 0.        , 0.        ],
       [0.        , 0.33333333, 0.        , 0.        , 0.33333333,0.        , 0.        , 0.33333333, 0.        ],
       [0.        , 0.        , 0.33333333, 0.        , 0.        ,0.33333333, 0.        , 0.        , 0.33333333]])

In [ ]:
pprint.pprint(vib_modes.translation_projector(4, 3))

In [ ]:
array([[0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  ,0.  ],
       [0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25,0.  ],
       [0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  ,0.25],
       [0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  ,0.  ],
       [0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25,0.  ],
       [0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  ,0.25],
       [0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  ,0.  ],
       [0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25,0.  ],
       [0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  ,0.25],
       [0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  ,0.  ],
       [0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25,0.  ],
       [0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  ,0.25]])

In [37]:
# No small tests in vib_modes.py for translation_projector.
# Supplementary comparison with course (not a course small test):
translation_projector(4,3)
np.testing.assert_allclose(
    translation_projector(4, 3),
    vib_modes.translation_projector(4, 3),
    atol=1e-7
)
np.testing.assert_allclose(
    translation_projector(2, 7),
    vib_modes.translation_projector(2, 7),
    atol=1e-7
)
print("translation_projector comparison passed!")

ValueError: too many values to unpack (expected 2)

### 1.2 `rotation_projector(vertices)`

Return a projector for the rotational degrees of freedom for vertices in 3D. Compute cross products between three orthogonal rotation axes and each vertex's position relative to the center of mass (assumed at the origin).

In [5]:
def rotation_projector(vertices, tol=1e-8):
    """Return projector of rotational degrees of freedom
    Inputs:
        vertices: (n, 3) np.array of vertices
        tol: tolerance for Gram-Schmidt (zero vector)
    Output:
        A (n * 3, n * 3) projector onto the span of rotational degrees of freedom
    """
    n = len(vertices)
    e_x = [1,0,0]
    e_y = [0,1,0]
    e_z = [0,0,1]
    d_x = []
    d_y = []
    d_z = []
    for m in range(0,n):
        d_x.append(np.cross(e_x, vertices[m]))
        d_y.append(np.cross(e_y, vertices[m]))
        d_z.append(np.cross(e_z, vertices[m]))
    #normalize
    if not np.allclose(np.linalg.norm(d_x), 0):
        d_x = d_x / np.linalg.norm(d_x)
    if not np.allclose(np.linalg.norm(d_y) ,0):
        d_y = d_y / np.linalg.norm(d_y)
    if not np.allclose(np.linalg.norm(d_z), 0):
        d_z = d_z / np.linalg.norm(d_z)
    print("gram-schmidt")
    print(d_x.shape)
    # gram-schmidt
    #d_x_o, _ = linalg.gram_schmidt(d_x, tol = tol)
    #d_y_o, _ = linalg.gram_schmidt(d_y, tol = tol)
    #d_z_o, _ = linalg.gram_schmidt(d_z, tol = tol)
    #print(d_x_o)
    # projector
    proj = np.zeros((n*3, n*3))
    proj = proj + (np.outer(d_x, d_x)) + (np.outer(d_y, d_y)) + (np.outer(d_z, d_z))
    return proj

In [9]:
print(vib_modes.rotation_projector(tetrahedron).shape)
print(vib_modes.rotation_projector(octahedron).shape)

(12, 12)
(18, 18)


In [ ]:
# No small tests in vib_modes.py for rotation_projector.
# Supplementary comparison with course (not a course small test):
np.testing.assert_allclose(
    rotation_projector(tetrahedron),
    vib_modes.rotation_projector(tetrahedron),
    atol=1e-7
)
np.testing.assert_allclose(
    rotation_projector(octahedron),
    vib_modes.rotation_projector(octahedron),
    atol=1e-7
)
print("rotation_projector comparison passed!")

### 1.3 `vibration_projector(vertices)`

Construct a projector for the vibrational degrees of freedom by subtracting the translational and rotational projectors from the identity.

In [6]:
def vibration_projector(vertices, tol=1e-8):
    """Build vibrational projector for vertices in 3D space.
    Inputs:
        vertices: (n, 3) vertices
        tol: tolerance for zero vector
    Output:
        (n * 3, n * 3) projector of vibrational modes
    """
    n = len(vertices)
    # YOUR CODE HERE
    return np.eye(n*3) - translation_projector(n,3) - rotation_projector(vertices)

In [ ]:
# No small tests in vib_modes.py for vibration_projector.
# Supplementary comparison with course (not a course small test):
np.testing.assert_allclose(
    vibration_projector(tetrahedron),
    vib_modes.vibration_projector(tetrahedron),
    atol=1e-7
)
np.testing.assert_allclose(
    vibration_projector(octahedron),
    vib_modes.vibration_projector(octahedron),
    atol=1e-7
)
print("vibration_projector comparison passed!")

In [10]:
print(vib_modes.vibration_projector(tetrahedron).shape)
print(vib_modes.vibration_projector(octahedron).shape)
vib_modes.vibration_projector(tetrahedron)

(12, 12)
(18, 18)


array([[ 5.00000000e-01,  1.25000000e-01, -1.25000000e-01,
         5.55111512e-17,  1.25000000e-01, -1.25000000e-01,
        -2.50000000e-01, -1.25000000e-01,  1.25000000e-01,
        -2.50000000e-01, -1.25000000e-01,  1.25000000e-01],
       [ 1.25000000e-01,  5.00000000e-01, -1.25000000e-01,
        -1.25000000e-01, -2.50000000e-01,  1.25000000e-01,
         1.25000000e-01,  5.55111512e-17, -1.25000000e-01,
        -1.25000000e-01, -2.50000000e-01,  1.25000000e-01],
       [-1.25000000e-01, -1.25000000e-01,  5.00000000e-01,
         1.25000000e-01,  1.25000000e-01, -2.50000000e-01,
         1.25000000e-01,  1.25000000e-01, -2.50000000e-01,
        -1.25000000e-01, -1.25000000e-01,  5.55111512e-17],
       [ 5.55111512e-17, -1.25000000e-01,  1.25000000e-01,
         5.00000000e-01, -1.25000000e-01,  1.25000000e-01,
        -2.50000000e-01,  1.25000000e-01, -1.25000000e-01,
        -2.50000000e-01,  1.25000000e-01, -1.25000000e-01],
       [ 1.25000000e-01, -2.50000000e-01,  1.2500000

---
## Section 2: Build Representations

The permutation representation encodes how symmetry operations permute the vertices. Combined with the vector representation and the vibrational projector, it yields the vibrational representation.

### 2.1 `permutation_representation(vertices, vec_rep)`

Given vertex coordinates and the group's 3D vector representation, return the permutation representation. Use a tolerance `eps` to match transformed vertices to original vertices.

In [7]:
def permutation_representation(vertices, vec_rep, eps=1e-4):
    """Return permutation representation of vertices under G with given vector representation.
    Inputs:
        vertices: (n, d) np.array of vertices
        vec_rep: (|G|, d, d) np.array of group representation on d-dimensional vector
        eps: absolute tolerance for matching vertices
    Output:
        (|G|, n, n) np.array of permutation representation of G on n vertices
    """
    n = len(vertices)
    g_l = len(vec_rep)
    d = len(vertices[0])
    perms = []
    for i in range(0, g_l):
        rep = vec_rep[i]
        perm = np.zeros((n,n))
        for j in range(0, n):
            new_vec = np.matmul(rep, vertices[j])
            for k in range(0, n):
                if np.allclose(vertices[k], new_vec, atol=eps):
                    perm[k][j] = 1
        perms.append(perm)
    return perms 

In [ ]:
# No small tests in vib_modes.py for permutation_representation.
# Supplementary comparison with course (not a course small test):
np.testing.assert_allclose(
    permutation_representation(tetrahedron, Td_vec),
    vib_modes.permutation_representation(tetrahedron, Td_vec),
    atol=1e-7
)
np.testing.assert_allclose(
    permutation_representation(octahedron, Oh_vec),
    vib_modes.permutation_representation(octahedron, Oh_vec),
    atol=1e-7
)
print("permutation_representation comparison passed!")

In [12]:
print(vib_modes.permutation_representation(octahedron, Td_vec).shape)
vib_modes.permutation_representation(tetrahedron, Td_vec)

(24, 6, 6)


array([[[0, 0, 1, 0],
        [0, 0, 0, 1],
        [1, 0, 0, 0],
        [0, 1, 0, 0]],

       [[1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 1, 0],
        [0, 0, 0, 1]],

       [[0, 0, 0, 1],
        [0, 0, 1, 0],
        [1, 0, 0, 0],
        [0, 1, 0, 0]],

       [[0, 0, 1, 0],
        [1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 0, 1]],

       [[1, 0, 0, 0],
        [0, 0, 0, 1],
        [0, 0, 1, 0],
        [0, 1, 0, 0]],

       [[0, 0, 1, 0],
        [1, 0, 0, 0],
        [0, 0, 0, 1],
        [0, 1, 0, 0]],

       [[1, 0, 0, 0],
        [0, 0, 1, 0],
        [0, 1, 0, 0],
        [0, 0, 0, 1]],

       [[0, 1, 0, 0],
        [0, 0, 1, 0],
        [1, 0, 0, 0],
        [0, 0, 0, 1]],

       [[0, 0, 1, 0],
        [0, 1, 0, 0],
        [1, 0, 0, 0],
        [0, 0, 0, 1]],

       [[1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 0, 1],
        [0, 0, 1, 0]],

       [[0, 0, 0, 1],
        [0, 0, 1, 0],
        [0, 1, 0, 0],
        [1, 0, 0, 0]],

       [[0

In [11]:
print(vib_modes.permutation_representation(tetrahedron, Td_vec).shape)
vib_modes.permutation_representation(tetrahedron, Td_vec)

(24, 4, 4)


array([[[0, 0, 1, 0],
        [0, 0, 0, 1],
        [1, 0, 0, 0],
        [0, 1, 0, 0]],

       [[1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 1, 0],
        [0, 0, 0, 1]],

       [[0, 0, 0, 1],
        [0, 0, 1, 0],
        [1, 0, 0, 0],
        [0, 1, 0, 0]],

       [[0, 0, 1, 0],
        [1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 0, 1]],

       [[1, 0, 0, 0],
        [0, 0, 0, 1],
        [0, 0, 1, 0],
        [0, 1, 0, 0]],

       [[0, 0, 1, 0],
        [1, 0, 0, 0],
        [0, 0, 0, 1],
        [0, 1, 0, 0]],

       [[1, 0, 0, 0],
        [0, 0, 1, 0],
        [0, 1, 0, 0],
        [0, 0, 0, 1]],

       [[0, 1, 0, 0],
        [0, 0, 1, 0],
        [1, 0, 0, 0],
        [0, 0, 0, 1]],

       [[0, 0, 1, 0],
        [0, 1, 0, 0],
        [1, 0, 0, 0],
        [0, 0, 0, 1]],

       [[1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 0, 1],
        [0, 0, 1, 0]],

       [[0, 0, 0, 1],
        [0, 0, 1, 0],
        [0, 1, 0, 0],
        [1, 0, 0, 0]],

       [[0

### 2.2 `vibration_representation(vertices, vec_rep)`

Compute the group representation on the vibrational subspace by combining the permutation representation, the vector representation (via tensor product), and the vibrational projector.

$\Gamma^{\text{a.s.}\otimes\text{vec}}$ = `kr_combo`

In [29]:
def vibration_representation(vertices, vec_rep, eps=1e-4, tol=1e-8):
    """Returns the group representation in the vibration subspace.
    Inputs:
        vertices: (n, 3) np.array of vertices
        vec_rep: (|G|, 3, 3) np.array of group representation on 3D vector
        eps: absolute tolerance for matching vertices
        tol: tolerance for zero vector
    Output:
        (|G|, n*3, n*3) np.array of vibration representation of G on n vertices in 3D space. Note that the flattened n*3 indices should be in the unflattened order (vertex index, spatial index).
    """
    perm_rep = vib_modes.permutation_representation(vertices, vec_rep, eps)
    g_l = len(vec_rep)
    kr_combo = []
    for i in range(0,g_l):
        kr_combo.append(np.kron(perm_rep[i], vec_rep[i]))
    kr_combo = rep.tensor_product(perm_rep, vec_rep)
    print(np.stack(kr_combo).shape) # (24,12,12)
    p_vib = vib_modes.vibration_projector(vertices, tol)
    print(p_vib.shape)
    q, p = linalg.gram_schmidt(p_vib, tol=tol)
    print(q.shape) #(6,12)
    vib_rep = []
    for i in range(0, g_l):
        vib_rep.append(np.matmul(p_vib.T, np.matmul(kr_combo[i], p_vib)))
    return vib_rep

In [30]:
# No small tests in vib_modes.py for vibration_representation.
# Supplementary comparison with course (not a course small test):
vib_rep_mine = vibration_representation(tetrahedron, Td_vec)
print(f"tetrahedron shape = {tetrahedron.shape}, Td_vec shape = {Td_vec.shape}, my result shape = {np.stack(vib_rep_mine).shape}")
vib_rep_course = vib_modes.vibration_representation(tetrahedron, Td_vec)
print(f"their result shape = {vib_rep_course.shape}")
np.testing.assert_allclose(
    vibration_representation(tetrahedron, Td_vec),
    vib_modes.vibration_representation(tetrahedron, Td_vec),
    atol=1e-7
)
print("vibration_representation comparison passed!")

(24, 12, 12)
(12, 12)
(6, 12)
tetrahedron shape = (4, 3), Td_vec shape = (24, 3, 3), my result shape = (24, 12, 12)
their result shape = (24, 12, 12)
(24, 12, 12)
(12, 12)
(6, 12)
vibration_representation comparison passed!


In [28]:
print(vib_modes.vibration_representation(tetrahedron, Td_vec).shape)
vib_modes.vibration_representation(tetrahedron, Td_vec)

(24, 12, 12)


array([[[ 2.50000000e-01, -1.25000000e-01, -1.25000000e-01, ...,
          0.00000000e+00,  1.25000000e-01,  1.25000000e-01],
        [-1.25000000e-01,  0.00000000e+00,  1.25000000e-01, ...,
          1.25000000e-01, -2.50000000e-01, -1.25000000e-01],
        [-1.25000000e-01,  1.25000000e-01,  2.50000000e-01, ...,
         -1.25000000e-01,  1.25000000e-01,  2.50000000e-01],
        ...,
        [ 6.93889390e-18,  1.25000000e-01, -1.25000000e-01, ...,
          2.50000000e-01, -1.25000000e-01,  1.25000000e-01],
        [ 1.25000000e-01, -2.50000000e-01,  1.25000000e-01, ...,
         -1.25000000e-01,  0.00000000e+00, -1.25000000e-01],
        [ 1.25000000e-01, -1.25000000e-01,  2.50000000e-01, ...,
          1.25000000e-01, -1.25000000e-01,  2.50000000e-01]],

       [[ 5.00000000e-01,  1.25000000e-01, -1.25000000e-01, ...,
         -2.50000000e-01, -1.25000000e-01,  1.25000000e-01],
        [ 1.25000000e-01,  5.00000000e-01, -1.25000000e-01, ...,
         -1.25000000e-01, -2.50000000e

---
## Section 3: Analyzing Vibrational Modes

### 3.1 `vibration_irrep_projectors(vertices, vec_rep, irreps)`

Decompose the vibrational projector into subspace projectors, each transforming according to an irreducible representation (irrep).

**Algorithm:**
1. Calculate the $3N \times 3N$ group representation on the vibrational subspace.
2. Infer a change of basis between each irrep and the vibrational representation.
3. Construct and stack the projectors corresponding to each change of basis.
4. Return a real array if the imaginary components are below `tol`.

In [ ]:
def vibration_irrep_projectors(vertices, vec_rep, irreps, eps=1e-4, tol=1e-8):
    """Returns the vibrational projector decomposed into subspace projectors that transform according to each irreducible representation (irrep).
    Inputs:
        vertices: (n, 3) np.array of vertices
        vec_rep: (|G|, 3, 3) np.array of group representation on 3D vector
        irreps:
        eps: absolute tolerance for matching vertices
        tol: tolerance for zero vector
    Output:
        np.array of irrep projectors of shape (num_irreps, n * 3, n * 3)
    """
    # YOUR CODE HERE
    pass

In [ ]:
# No small tests in vib_modes.py for vibration_irrep_projectors.
# Supplementary check (not a course small test):
projs = vibration_irrep_projectors(tetrahedron, Td_vec, Td_irreps)
# Each projector should be idempotent (P^2 = P)
for i, p in enumerate(projs):
    np.testing.assert_allclose(p @ p, p, atol=1e-7, err_msg=f"Projector {i} not idempotent")
# Projectors should sum to the vibration projector
np.testing.assert_allclose(
    projs.sum(axis=0),
    vib_modes.vibration_projector(tetrahedron),
    atol=1e-7,
)
print("vibration_irrep_projectors check passed!")

### 3.2 Which irreps?

Use `vib_modes.vibration_irrep_projectors`, `rep.character_table`, and the provided irreps to answer questions about the vibrational modes of a tetrahedron and an octahedron.

For reference, check the conventional character tables:
- [$T_d$](http://symmetry.jacobs-university.de/cgi-bin/group.cgi?group=902&option=4)
- [$O_h$](http://symmetry.constructor.university/cgi-bin/group.cgi?group=904&option=4)

The helper `vib_modes.conjugacy_class_in_table_notation` converts between 3D vector representations and standard notation for symmetry operations (e.g., $C_n$, $\sigma$, $S_n$, $E$, $i$).

In [ ]:
# Explore which irreps the vibrational modes transform as.
# Compute vibration_irrep_projectors for the tetrahedron and octahedron,
# then check which projectors have nonzero rank.
#
# Useful: np.trace(projector) gives the rank (= dimension of the subspace).
# Also try: rep.character_table(irreps, groups.conjugacy_classes(table))
# and vib_modes.conjugacy_class_in_table_notation(vec_rep, conj_classes)

# YOUR EXPLORATION HERE

In [ ]:
# YOUR ANSWER: Which irreps do the vibrational modes of a tetrahedron transform as?
# Options: A_1, A_2, E, T_1, T_2
# tetra_mode_irreps = [...]  # list of irrep labels

In [ ]:
# YOUR ANSWER: Which irreps do the vibrational modes of an octahedron transform as?
# Options: A_{1g}, A_{2g}, E_g, T_{1g}, T_{2g}, A_{1u}, A_{2u}, E_u, T_{1u}, T_{2u}
# oct_mode_irreps = [...]  # list of irrep labels

### 3.3 Infrared and Raman Activity

**Infrared activity** arises from changes in the dipole moment, which transforms as a 3D vector $(x, y, z)$.

**Raman activity** is related to changes in polarizability, which transforms as the symmetric part of a $3 \times 3$ matrix: $(xy, yz, zx, 2z^2 - x^2 - y^2, x^2 - y^2)$.

Use the character table to determine which irreps are IR-active and Raman-active.

In [ ]:
# Explore IR and Raman activity.
# An irrep is IR-active if the tensor product of the irrep with the
# vector representation contains the trivial representation.
# An irrep is Raman-active if the tensor product with the symmetric
# rank-2 tensor representation contains the trivial representation.

# YOUR EXPLORATION HERE

In [ ]:
# YOUR ANSWER: Which Td irreps are infrared active?
# Options: A_1, A_2, E, T_1, T_2
# tetra_ir_modes = [...]

In [ ]:
# YOUR ANSWER: Which Td irreps are Raman active?
# Options: A_1, A_2, E, T_1, T_2
# tetra_raman_modes = [...]

In [ ]:
# YOUR ANSWER: Which Oh irreps are infrared active?
# Options: A_{1g}, A_{2g}, E_g, T_{1g}, T_{2g}, A_{1u}, A_{2u}, E_u, T_{1u}, T_{2u}
# oct_ir_modes = [...]

In [ ]:
# YOUR ANSWER: Which Oh irreps are Raman active?
# Options: A_{1g}, A_{2g}, E_g, T_{1g}, T_{2g}, A_{1u}, A_{2u}, E_u, T_{1u}, T_{2u}
# oct_raman_modes = [...]

### 3.4 `vibrational_modes(vertices, vec_rep, irreps)`

Take the projectors from `vibration_irrep_projectors` and return an orthogonal basis for each irrep subspace. Each projector may contain multiple copies of an irrep — its rank is $n_i \times d_i$ where $n_i$ is the multiplicity and $d_i$ is the irrep dimension.

You can visualize modes with `vib_modes.plot_vibrational_mode(vertices, mode.reshape(-1, 3))`.

In [ ]:
def vibrational_modes(vertices, vec_rep, irreps, eps=1e-4, tol=1e-8):
    """Returns an orthogonal basis for each vibration subspace projectors that transform as specific irreps.
    Inputs:
        vertices: (n, 3) np.array of vertices
        vec_rep: (|G|, 3, 3) np.array of group representation on 3D vector
        irreps:
        eps: absolute tolerance for matching vertices
        tol: tolerance for zero vector
    Output:
        list of np.arrays with shape (irrep proj rank, n * 3) that are orthogonal bases for each irrep subspace
    """
    # YOUR CODE HERE
    pass

In [14]:
vib_modes.vibration_irrep_projectors(tetrahedron, Td_vec, Td_irreps)

array([[[ 8.33333333e-02,  8.33333333e-02, -8.33333333e-02,
          8.33333333e-02, -8.33333333e-02,  8.33333333e-02,
         -8.33333333e-02,  8.33333333e-02,  8.33333333e-02,
         -8.33333333e-02, -8.33333333e-02, -8.33333333e-02],
        [ 8.33333333e-02,  8.33333333e-02, -8.33333333e-02,
          8.33333333e-02, -8.33333333e-02,  8.33333333e-02,
         -8.33333333e-02,  8.33333333e-02,  8.33333333e-02,
         -8.33333333e-02, -8.33333333e-02, -8.33333333e-02],
        [-8.33333333e-02, -8.33333333e-02,  8.33333333e-02,
         -8.33333333e-02,  8.33333333e-02, -8.33333333e-02,
          8.33333333e-02, -8.33333333e-02, -8.33333333e-02,
          8.33333333e-02,  8.33333333e-02,  8.33333333e-02],
        [ 8.33333333e-02,  8.33333333e-02, -8.33333333e-02,
          8.33333333e-02, -8.33333333e-02,  8.33333333e-02,
         -8.33333333e-02,  8.33333333e-02,  8.33333333e-02,
         -8.33333333e-02, -8.33333333e-02, -8.33333333e-02],
        [-8.33333333e-02, -8.3333333

In [ ]:
# No small tests in vib_modes.py for vibrational_modes.
# Supplementary check (not a course small test):
modes = vibrational_modes(tetrahedron, Td_vec, Td_irreps)
projs_course = vib_modes.vibration_irrep_projectors(tetrahedron, Td_vec, Td_irreps)
# Each mode set should be orthonormal and span the same space as the projector
for i, (m, p) in enumerate(zip(modes, projs_course)):
    if len(m) > 0:
        np.testing.assert_allclose(m @ m.T, np.eye(len(m)), atol=1e-7,
            err_msg=f"Modes for irrep {i} not orthonormal")
        np.testing.assert_allclose(
            linalg.gram_schmidt(m)[1], p, atol=1e-7,
            err_msg=f"Modes for irrep {i} don't span the correct subspace")
print("vibrational_modes check passed!")

In [ ]:
# Visualize vibrational modes of the tetrahedron
# modes_Td = vib_modes.vibrational_modes(tetrahedron, Td_vec, Td_irreps)
# for i, m in enumerate(modes_Td):
#     for j, mode in enumerate(m):
#         vib_modes.plot_vibrational_mode(
#             tetrahedron, mode.reshape(-1, 3),
#             title=f"Irrep {i}, mode {j}"
#         ).show()

---
## Section 4: Preserving Subgroup Symmetry

When a system undergoes a phase transition or distortion, its symmetry often reduces to a subgroup. Branching rules describe how the irreps of the full group decompose into those of the subgroup.

### 4.1 `isomorphic_subgroups(group_table, subgroup_table)`

Find all subgroups of a group that are isomorphic to a given candidate subgroup. For each match, return the subset of group elements and the list of valid isomorphisms.

**Steps:**
1. Generate candidate subgroups with `groups_fast.generate_subgroups_dynamic_programming`
2. Filter by size, compute subtables with `groups.subgroup_table_from_group_table` and `groups.remap_to_minimal`
3. Find isomorphisms with `groups_fast.isomorphisms_generator_backtracking`

In [ ]:
def isomorphic_subgroups(group_table, subgroup_table):
    """Identify isomorphic subgroups between a group and a candidate subgroup.

    This function takes in a table representing the full group and a table for a candidate subgroup,
    and determines for each subgroup the subset of group elements that form it and a list of valid
    isomorphisms between the subgroup and the candidate subgroup.

    Inputs:
    group_table: multiplication table of group, np.array of shape (|G|, |G|)
    subgroup_table: multiplication table of subgroup, np.array of shape (|H|, |H|)

    Returns:
      a list of tuples. For each tuple:
            - The first element is list of the subset of elements from the group that form the subgroup.
            - The second element is a list of valid isomorphisms mapping this subgroup to the candidate subgroup.
    """
    subgroups = groups_fast.generate_subgroups_dynamic_programming(group_table)
    
    # YOUR CODE HERE
    pass

In [ ]:
# No small tests in vib_modes.py for isomorphic_subgroups.
# Supplementary comparison with course (not a course small test):
result = isomorphic_subgroups(Td_table, C2v_table)
course_result = vib_modes.isomorphic_subgroups(Td_table, C2v_table)
# Compare the sets of subgroup element sets
result_sets = set(frozenset(elems) for elems, _ in result)
course_sets = set(frozenset(elems) for elems, _ in course_result)
assert result_sets == course_sets, f"Subgroup sets differ: got {len(result_sets)}, expected {len(course_sets)}"
# Compare number of isomorphisms per subgroup
result_iso_counts = sorted(len(isos) for _, isos in result)
course_iso_counts = sorted(len(isos) for _, isos in course_result)
assert result_iso_counts == course_iso_counts, f"Isomorphism counts differ"
print("isomorphic_subgroups comparison passed!")

### 4.2 `branching_change_of_basis(G_irreps, H_irreps, H_elements)`

Compute change-of-basis matrices between the irreps of group $G$ and subgroup $H$ according to the branching rules.

**Steps:**
1. For each G irrep, restrict it to the subgroup elements: `G_ir[H_elements]`
2. For each H irrep, compute the change of basis with `linalg.infer_change_of_basis`
3. Return a nested list: `result[i][j]` = change-of-basis from G irrep $i$ to H irrep $j$

In [ ]:
def branching_change_of_basis(G_irreps, H_irreps, H_elements, tol=1e-6):
    """
    Compute the change-of-basis matrices between the irreducible representations (irreps) of a group G
    and those of a subgroup H according to the branching rules.

    This function assumes:
      - G_irreps is a list of NumPy arrays, each of shape (|G|, d, d), representing an irrep of G
        in a particular basis.
      - H_irreps is a list of NumPy arrays, each of shape (|H|, d, d), representing the corresponding
        irreps of H in a chosen basis.
      - H_elements is an ordered collection (e.g., list or array) of elements from G that form the subgroup H,
        arranged in the same order as the H_irreps.

    The function returns a list of change-of-basis matrices that map the basis of each irrep of G
    to the corresponding basis in which the irrep of H is expressed.

    Inputs:
      G_irreps : list of np.ndarray
          A list of irreducible representation matrices of group G. Each array has shape (|G|, d, d), where d
          is the dimension of that irrep.
      H_irreps : list of np.ndarray
          A list of irreducible representation matrices of subgroup H. Each array has shape (|H|, d, d).
      H_elements : list or array-like
          An ordered collection of elements of G that form the subgroup H, matching the order of the H_irreps.

    Output:
      list of list of np.ndarray
          A nested list where each outer list entry corresponds to a G irrep, and each inner list contains the
          change-of-basis matrices that transform the basis of the corresponding G irrep into the bases of the H irreps.
    """
    # YOUR CODE HERE
    pass

In [16]:
# No small tests in vib_modes.py for branching_change_of_basis.
# Supplementary comparison with course (not a course small test):
# Use one C2v subgroup of Td for testing
_subs_elements = [10, 6, 1, 23]
result = branching_change_of_basis(Td_irreps, C2v_irreps, _subs_elements)
course_result = vib_modes.branching_change_of_basis(Td_irreps, C2v_irreps, _subs_elements)
for i, (r_row, c_row) in enumerate(zip(result, course_result)):
    for j, (r, c) in enumerate(zip(r_row, c_row)):
        assert r.shape == c.shape, (
            f"Shape mismatch for G_irrep {i}, H_irrep {j}: got {r.shape}, expected {c.shape}"
        )
print("branching_change_of_basis comparison passed!")

NameError: name 'branching_change_of_basis' is not defined

In [17]:
vib_modes.branching_change_of_basis(Td_irreps, C2v_irreps, _subs_elements)

[[array([], shape=(0, 1, 1), dtype=complex128),
  array([], shape=(0, 1, 1), dtype=complex128),
  array([], shape=(0, 1, 1), dtype=complex128),
  array([[[1.+0.j]]])],
 [array([], shape=(0, 1, 2), dtype=complex128),
  array([[[0.78105795+0.j, 0.62445855+0.j]]]),
  array([], shape=(0, 1, 2), dtype=complex128),
  array([[[ 0.62445855+0.j, -0.78105795+0.j]]])],
 [array([], shape=(0, 1, 1), dtype=complex128),
  array([[[1.+0.j]]]),
  array([], shape=(0, 1, 1), dtype=complex128),
  array([], shape=(0, 1, 1), dtype=complex128)],
 [array([[[0.81042266+0.j, 0.55942708+0.j, 0.17394383+0.j]]]),
  array([], shape=(0, 1, 3), dtype=complex128),
  array([[[ 0.53386915+0.j, -0.82748249+0.j,  0.17394383+0.j]]]),
  array([[[ 0.24124436+0.j, -0.04810478+0.j, -0.96927142+0.j]]])],
 [array([[[ 0.34087031+0.00000000e+00j, -0.90365773+3.69242724e-16j,
           -0.25924917+1.79487017e-16j]]]),
  array([[[ 0.33929316+0.00000000e+00j, -0.13892592+4.71932291e-17j,
            0.93036538-6.10754701e-16j]]]),
 

### 4.3 Branching rules of $T_d$ irreps under $C_{2v}$

Use `vib_modes.branching_change_of_basis` to determine how each $T_d$ irrep decomposes under $C_{2v}$. A nonzero change-of-basis matrix for a given $(G\text{-irrep}, H\text{-irrep})$ pair means that $H$-irrep appears in the branching of that $G$-irrep.

In [ ]:
# Explore branching rules.
# 1. Use isomorphic_subgroups (or vib_modes.isomorphic_subgroups) to find
#    a C2v subgroup of Td
# 2. Use branching_change_of_basis (or vib_modes.branching_change_of_basis)
#    to compute the branching rules
# 3. Check which change-of-basis matrices are nonzero

# YOUR EXPLORATION HERE

In [ ]:
# YOUR ANSWERS: How does each Td irrep branch under C2v?
# Options for each: A_1, A_2, B_1, B_2
#
# A1_to_C2v = [...]   # e.g. ["A_1"]
# A2_to_C2v = [...]
# E_to_C2v = [...]
# T1_to_C2v = [...]
# T2_to_C2v = [...]

---
## Explore Further

In [ ]:
# Try analyzing a different molecule or point group!

In [ ]:
# Visualize vibrational modes